# S4 · GRPO: the group replaces the value model

Study path stage 4 of 7 · Chapter 13 · Day 22 · **DGX Spark GPU first**

In S3 you built PPO: a clipped surrogate, a KL penalty and a learned value model that turns one
reward per answer into one advantage per token. **Group-relative policy optimization (GRPO)** keeps
the clipped surrogate and the KL term and removes the value model. It samples $G$ answers to the same
prompt and uses each answer's reward relative to its own group as the advantage. After this notebook
you'll be able to write GRPO's advantage, its three common loss normalisations, its two KL
estimators and its full loss from scratch, say exactly what each choice does to the gradient, and run
GRPO on Qwen3-0.6B with only two networks on the GPU: the policy and a frozen reference.

| Part | Question | Discussions |
|---|---|---|
| A · Group advantages | What does the per-group z-score do, and how often does a group give no signal at all? | [D21, D32, D33](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| B · Loss normalisation | Per-answer mean, token mean or a constant: how much is each token pushed, and why does length matter? Why divide by the std? | [D36, D38](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| C · KL estimators | k1 and k3: what does each estimate, with what variance, and what gradient does k3 give in the loss? | [D35](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| D · The GRPO loss | Clipped surrogate + β·k3, written once; what happens with one update vs several on one batch? | [D35, D36](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| E · Toy GRPO vs REINFORCE | Does the group baseline beat no baseline on a 0/1 reward, and when does a baseline matter? | [D38, D39](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| F · GRPO on Qwen3-0.6B | Reward, KL, zero-variance groups and length over 40 real updates; memory without a value model | [D21, D33, D34](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |

RL terms (state, action, return, value, advantage, baseline, credit assignment, importance ratio,
clipping) are defined in the learning note's glossary, [RL terms in the LLM setting](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md#-rl-terms-in-the-llm-setting-glossary-d39).
Here an **action** is one generated token and a **state** is the prompt plus the tokens so far.

**How the exercises work.** Each exercise has the same rhythm:

1. **Read the task**, then write your lines where you see `...` in the exercise cell.
2. **Run the check.** It tests your code on small cases and prints ✅ or ❌ with a hint. Nothing
   crashes if you haven't finished; you'll see ⏳ instead.
3. **Compare with the reference**, which follows each check and then runs a small experiment.

Later cells use your version if its check passed, otherwise the reference, via
`mine_or_ref(globals(), …)`, so one stuck exercise never blocks the rest. Predict cells ask you to
write an answer before running the reference.

**Data.** Single-turn user questions from [Anthropic HH-RLHF](https://huggingface.co/datasets/Anthropic/hh-rlhf)
`helpful-base` (MIT license), train split from row 20,000 for training and the test split for
evaluation. Only the questions are used; no preference labels.

**Environment.** Kernel `Python (DGX Spark Native)` with CUDA, and `Qwen/Qwen3-0.6B` plus the dataset
in the local Hugging Face cache; nothing is downloaded. Checks and references come from the tested
module `dongxi_llms.study_path.s4_grpo`.

**Run time.** About 2 minutes on an idle Spark (the reference run took 116 s): Parts A–E take about
20 s, Part F about 1.5 minutes of training plus model loading and two evaluations. Peak GPU memory
17.2 GiB.

**Scale and limits.** Parts A–D are exact calculations on fixed tensors. Part E is a tabular policy
(16 tasks, 3-token answers), averaged over 8 seeds. Part F is one 40-update run of a 0.6B model
against a rule reward (answer length), with a held-out before/after comparison. It verifies the
mechanics and shows the diagnostics move as the theory says; it doesn't measure helpfulness,
correctness or anything about larger models, and a length reward is easy to satisfy with generic
text (we'll see that).

In [ ]:
import inspect, math, random, sys, textwrap, time
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(ROOT / "src"))               # the course package, wherever this copy lives
import matplotlib.pyplot as plt
import numpy as np
import torch
from dongxi_llms.study_path import qwen as Q
from dongxi_llms.study_path import s4_grpo as s4
from dongxi_llms.study_path.common import mine_or_ref

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
random.seed(0); np.random.seed(0); torch.manual_seed(0)
BLUE, ORANGE, GRAY, GREEN, RED = "#2563EB", "#EA580C", "#94A3B8", "#16A34A", "#DC2626"
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

---

# Part A · Group advantages

PPO needs a value model to say what reward an answer "should" get, so it can tell better-than-expected
from worse-than-expected (S3, D22–D25). GRPO gets that reference level from sampling instead. For one
prompt $x$ it samples a **group** of $G$ answers $y_1,\dots,y_G$ from the current policy, scores each
with a reward $R_i$, and sets the **group-relative advantage**

$$
\hat A_i=\frac{R_i-\bar R}{s+\epsilon},\qquad
\bar R=\frac1G\sum_{j=1}^G R_j,\qquad
s=\sqrt{\frac1G\sum_{j=1}^G (R_j-\bar R)^2},\qquad \epsilon=10^{-8}.
$$

$s$ is the **population** standard deviation (divide by $G$, `correction=0` in PyTorch), as in
Chapter 13 §13.1. The group mean $\bar R$ plays the role of the **baseline**; dividing by $s$ is an
extra step that PPO doesn't have. Every token of answer $i$ receives the same $\hat A_i$: GRPO does no
per-token credit assignment (D38).

Two properties follow directly. The advantages of a group sum to zero. And if all $G$ rewards are
equal, every $\hat A_i$ is zero: the group gives no policy-gradient signal. We call it a
**zero-variance group**. With a 0/1 reward that happens whenever a prompt is solved by all $G$
answers or by none.

### Prediction

## Exercise 1 · the per-group z-score

Write GRPO's advantage. `R` has shape `[P, G]`: one row per prompt, one column per sampled answer.
Two lines: the centred rewards, and the population std of each row (with `keepdim=True`, so it
broadcasts against `[P, G]`). The return line is given; it returns exactly 0 for a constant group.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_group_advantages(R, eps=1e-8):
    centered = ...      # R minus the mean of its own row, [P, G]
    std = ...           # population std of each row (correction=0), [P, 1]
    return torch.where(std > 0, centered / (std + eps), torch.zeros_like(centered))

In [ ]:
ex1_ok = s4.check_group_advantages(globals().get("my_group_advantages"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def group_advantages_ref(R, eps=1e-8):
    centered = R - R.mean(-1, keepdim=True)
    std = R.std(-1, correction=0, keepdim=True)
    return torch.where(std > 0, centered / (std + eps), torch.zeros_like(centered))

s4.check_group_advantages(group_advantages_ref, strict=True)
group_advantages = mine_or_ref(globals(), "my_group_advantages", ex1_ok, group_advantages_ref)

R_demo = torch.tensor([[0., 0., 1., 1.], [0., 0., 0., 1.], [1., 1., 1., 1.], [0., 0., 0., 0.]])
A_demo = group_advantages(R_demo)
for r, a in zip(R_demo, A_demo):
    print(f"R = {r.tolist()}  →  Â = {[round(v, 3) for v in a.tolist()]}   sum {a.sum():+.1e}")
print("sample std instead of population std on [0,0,1,1]:",
      ((R_demo[0] - R_demo[0].mean()) / R_demo[0].std()).tolist())

### Reference explanation

### Reference answer and mechanism

For $[0,0,1,1]$: $\bar R=0.5$, $s=0.5$, so $\hat A=[-1,-1,+1,+1]$. For $[0,0,0,1]$: $\bar R=0.25$,
$s=\sqrt{0.25\cdot0.75}=0.433$, so the single success gets $+1.732$ and each failure $-0.577$. A
rare success receives a large advantage; Part B returns to this. The two constant groups give exactly
zero.

The sample std (`correction=1`, divide by $G-1$) is larger by $\sqrt{G/(G-1)}$, so the advantages
shrink: $\pm0.866$ instead of $\pm1$ at $G=4$. That is a change of update scale, not of direction.

Because $\hat A$ subtracts the group mean and divides by the group std, replacing every reward by
$aR+b$ with $a>0$ leaves $\hat A$ unchanged. Part E uses this: the GRPO update doesn't depend on the
offset or the scale of the reward.

### Prediction

## Exercise 2 · how often a group gives no signal

Assume a 0/1 reward and that each of the $G$ answers succeeds independently with probability $p$.
Write the probability that the group is constant (zero variance). One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_zero_signal_prob(p, G):
    return ...          # P(all G answers fail) + P(all G answers succeed)

In [ ]:
ex2_ok = s4.check_zero_signal_prob(globals().get("my_zero_signal_prob"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def zero_signal_prob_ref(p, G):
    return (1 - p) ** G + p ** G

s4.check_zero_signal_prob(zero_signal_prob_ref, strict=True)
zero_signal_prob = mine_or_ref(globals(), "my_zero_signal_prob", ex2_ok, zero_signal_prob_ref)

gen = torch.Generator().manual_seed(0)
for p, G in [(0.1, 4), (0.1, 8), (0.5, 8), (0.9, 8)]:
    R = (torch.rand(20000, G, generator=gen) < p).float()
    simulated = (R.std(-1) == 0).float().mean().item()
    print(f"p = {p}, G = {G}:  formula {zero_signal_prob(p, G):.4f}  |  simulated over 20,000 groups {simulated:.4f}")

### Reference explanation

### Reference answer and mechanism

$P(\text{zero variance})=(1-p)^G+p^G$. At $p=0.1$: $0.6562$ for $G=4$ and $0.4305$ for $G=8$
(Chapter 13 §13.2 has the same numbers). The simulation agrees to about $\pm0.004$, the Monte Carlo
error with 20,000 groups. Larger groups make a mixed group more likely, at the cost of generating
more answers per prompt.

### Prediction

## Predict 1 · a well-trained policy

A batch has 64 prompts with $G=8$ answers each. The policy now solves every prompt with probability
$p=0.9$ per answer. How many of the 64 groups give no policy-gradient signal? If you discard those
groups and keep sampling until you have 64 groups that do give a signal (DAPO's *dynamic sampling*),
about how many groups, and answers, do you have to generate?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
p, G, B = 0.9, 8, 64
p0 = zero_signal_prob(p, G)
print(f"P(zero-variance group) = 0.9^8 + 0.1^8 = {p0:.4f}  →  expected {B * p0:.1f} of {B} groups give no signal")
print(f"   of which all-success: {p ** G:.4f}, all-failure: {(1 - p) ** G:.1e}")
print(f"to keep {B} informative groups: about {B / (1 - p0):.0f} groups = {B * G / (1 - p0):.0f} answers "
      f"instead of {B * G}")
gen = torch.Generator().manual_seed(1)
counts = np.array([((torch.rand(B, G, generator=gen) < p).float().std(-1) == 0).sum().item() for _ in range(2000)])
print(f"simulated over 2,000 batches: mean {counts.mean():.1f}, 5–95% range {np.percentile(counts, 5):.0f}–"
      f"{np.percentile(counts, 95):.0f} zero-variance groups")

### Reference explanation

### Reference answer and mechanism

$0.9^8+0.1^8=0.4305$, so about **27.5 of 64 groups** (simulated 5–95% range 21–34) have all eight
answers correct and contribute nothing to the policy-gradient term. Only the KL term still acts on
them. As training succeeds, $p$ rises and the useful part of the batch shrinks: the signal disappears
because the policy solves the prompts, not because anything broke.

Discarding those groups and resampling keeps the batch full of mixed groups but costs generation:
about $64/(1-0.4305)\approx112$ groups, i.e. about 900 answers instead of 512. That is the trade-off
behind DAPO's dynamic sampling ([Yu et al., 2025](https://arxiv.org/abs/2503.14476)), and why Chapter
13 §13.5.2 insists on counting the rejected groups. The same arithmetic at $p=0.1$ gives the same
number of zero-variance groups, now all failures.

In [ ]:
ps = np.linspace(0.005, 0.995, 199)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.7))
gen = torch.Generator().manual_seed(2)
for G, c in zip([2, 4, 8, 16, 32], [GRAY, ORANGE, BLUE, GREEN, RED]):
    axes[0].plot(ps, zero_signal_prob(ps, G), color=c, label=f"G = {G}")
    axes[1].plot(ps, 1 / (1 - zero_signal_prob(ps, G)), color=c, label=f"G = {G}")
p_sim = np.arange(0.05, 1.0, 0.1)
sim = [((torch.rand(4000, 8, generator=gen) < p).float().std(-1) == 0).float().mean().item() for p in p_sim]
axes[0].scatter(p_sim, sim, color=BLUE, s=18, zorder=3, label="G = 8, simulated")
axes[0].axvline(0.9, color=GRAY, ls=":")
axes[0].annotate("p = 0.9, G = 8: 0.43", (0.9, 0.43), (0.42, 0.30), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color=GRAY))
axes[0].set(xlabel="success probability p per answer", ylabel="P(group has zero variance)",
            title="Groups that give no policy-gradient signal")
axes[1].set_yscale("log"); axes[1].set_ylim(1, 200)
axes[1].set(xlabel="success probability p per answer", ylabel="groups sampled per informative group",
            title="Cost of keeping only mixed groups")
axes[0].legend(fontsize=8, frameon=False, loc="upper center", ncol=2)
axes[1].legend(fontsize=8, frameon=False, loc="upper center", ncol=2)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Probability that a group of G answers has zero reward variance, and the sampling cost of discarding such groups](../figures/chapter-13/study-path-s4_grpo-01.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part B · How the loss is normalised

The GRPO loss is a sum of per-token terms $\ell_{it}$ (Part D writes them out). Answers have different
lengths $|o_i|$, so the batch is a padded `[N, T]` tensor with a mask $m_{it}$ that is 1 on generated
tokens (through `<|im_end|>`) and 0 on padding. Turning the terms into one scalar needs a
normalisation, and the choice decides how strongly each token is pushed. Three are in use:

$$
L_{\text{seq}}=\frac1N\sum_{i=1}^N\frac1{|o_i|}\sum_t m_{it}\ell_{it}
\qquad
L_{\text{tok}}=\frac{\sum_{i,t}m_{it}\ell_{it}}{\sum_i |o_i|}
\qquad
L_{\text{const}}=\frac{\sum_{i,t}m_{it}\ell_{it}}{N\cdot L_{\max}}
$$

$L_{\text{seq}}$ is the original GRPO ([DeepSeekMath, Shao et al., 2024](https://arxiv.org/abs/2402.03300)):
average each answer over its own tokens, then average answers. $L_{\text{tok}}$ is DAPO's token-level
loss: one average over every real token in the batch. $L_{\text{const}}$ is Dr. GRPO
([Liu et al., 2025](https://arxiv.org/abs/2503.20783)): divide by a constant, the number of answers
times the generation budget $L_{\max}$. The **per-token weight** $w_{it}=\partial L/\partial\ell_{it}$
is $\frac{1}{N|o_i|}$, $\frac{1}{\sum_j|o_j|}$ and $\frac{1}{N L_{\max}}$ respectively.

### Prediction

## Exercise 3 · three normalisations

Write the three aggregations. `u` holds per-token terms and `m` the mask, both `[N, T]`; padding
positions of `u` can hold any value, so multiply by `m` before summing. One line each.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_seq_mean(u, m):
    return ...          # average over each row's real tokens, then over rows   (GRPO)

def my_token_mean(u, m):
    return ...          # one average over all real tokens in the batch          (DAPO)

def my_constant_norm(u, m, L_max):
    return ...          # masked sum divided by N · L_max                        (Dr. GRPO)

In [ ]:
ex3_ok = s4.check_aggregations(globals().get("my_seq_mean"), globals().get("my_token_mean"),
                               globals().get("my_constant_norm"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def seq_mean_ref(u, m):
    return ((u * m).sum(1) / m.sum(1)).mean()

def token_mean_ref(u, m):
    return (u * m).sum() / m.sum()

def constant_norm_ref(u, m, L_max):
    return (u * m).sum() / (u.shape[0] * L_max)

s4.check_aggregations(seq_mean_ref, token_mean_ref, constant_norm_ref, strict=True)
seq_mean = mine_or_ref(globals(), "my_seq_mean", ex3_ok, seq_mean_ref)
token_mean = mine_or_ref(globals(), "my_token_mean", ex3_ok, token_mean_ref)
constant_norm = mine_or_ref(globals(), "my_constant_norm", ex3_ok, constant_norm_ref)

# per-token weights by autograd: three answers of 5, 20 and 80 tokens, generation budget 80
lengths, L_MAX = torch.tensor([5, 20, 80]), 80
m3 = (torch.arange(L_MAX)[None] < lengths[:, None]).double()
weights = {}
for name, fn in [("seq-mean (GRPO)", seq_mean), ("token-mean (DAPO)", token_mean),
                 ("constant (Dr. GRPO)", lambda u, m: constant_norm(u, m, L_MAX))]:
    u = torch.zeros(3, L_MAX, dtype=torch.float64, requires_grad=True)
    fn(u, m3).backward()
    weights[name] = u.grad[:, 0].clone()
    print(f"{name:20s} per-token weight {[f'{w:.5f}' for w in u.grad[:, 0].tolist()]}  "
          f"answer total {[f'{w:.3f}' for w in u.grad.sum(1).tolist()]}")

### Reference explanation

### Reference answer and mechanism

Under the per-answer mean each answer has the same total weight $1/N=0.333$, spread over its own
tokens: a token of the 5-token answer weighs $1/15=0.0667$, a token of the 80-token answer
$1/240=0.0042$, 16 times less. Under the token mean every token weighs $1/105$, so the long answer
carries $80/105=76\%$ of the batch. The constant gives every token $1/240$; the totals are
proportional to length, as with the token mean, but the divisor no longer depends on which answers
happened to be sampled. All three put zero weight on padding.

### Prediction

## Predict 2 · the length bias of the per-answer mean

A batch of $N=4$ answers: two wrong ones with $\hat A=-1$, of 10 and 100 tokens, and two correct
ones with $\hat A=+1$, also of 10 and 100 tokens. Take one fresh update, so $\rho=1$ and the
surrogate's gradient with respect to a token's log-probability is $-w_{it}\hat A_i$. Under the
per-answer mean, how hard is each token of the **long wrong** answer pushed down compared with each
token of the short wrong one? And under the token mean? What does that suggest for the length of
wrong answers over many updates?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
lengths, A4 = torch.tensor([10, 100, 10, 100]), torch.tensor([-1., -1., 1., 1.], dtype=torch.float64)
m4 = (torch.arange(100)[None] < lengths[:, None]).double()
for name, agg in [("seq-mean", seq_mean), ("token-mean", token_mean), ("constant", lambda u, m: constant_norm(u, m, 100))]:
    logp = torch.zeros(4, 100, dtype=torch.float64, requires_grad=True)
    agg(-(torch.exp(logp - logp.detach()) * A4[:, None]), m4).backward()     # surrogate at ρ = 1
    g = logp.grad[:, 0]
    print(f"{name:10s} ∂L/∂logp per token: wrong-10 {g[0]:+.5f}  wrong-100 {g[1]:+.5f}  "
          f"correct-10 {g[2]:+.5f}  correct-100 {g[3]:+.5f}   long/short = {g[1] / g[0]:.2f}")

### Reference explanation

### Reference answer and mechanism

A positive $\partial L/\partial\log\pi$ means gradient descent lowers that token's log-probability.
Under the per-answer mean each token of the 100-token wrong answer is pushed down with
$1/(4\cdot100)=0.0025$, ten times less than a token of the 10-token wrong answer ($0.025$). For
correct answers it is the same factor in the other direction: short correct answers get the larger
per-token push up. So a wrong answer is penalised less per token the longer it is.
[Liu et al. (2025)](https://arxiv.org/abs/2503.20783) argue that this lets wrong answers grow longer
during training and propose the constant normaliser. Under the token mean and the constant, every token
gets the same magnitude ($1/220$ and $1/400$), whatever the length of its answer.

These are exact per-token coefficients at $\rho=1$. Whether lengths actually drift in a run also
depends on how often long answers are wrong, on truncation and on shared parameters; that is a
separate measurement (Chapter 14 and S7).

Dr. GRPO makes a second change: it drops the division by the group std. To see what that division
does, take a 0/1 reward with $k$ successes out of $G$, so $\hat p=k/G$. The group std is
$s=\sqrt{\hat p(1-\hat p)}$, the centred advantages are $1-\hat p$ (successes) and $-\hat p$
(failures), and the total advantage magnitude of the group is

$$
\sum_i|R_i-\bar R|=2G\hat p(1-\hat p)\quad\text{(centred)},\qquad
\sum_i|\hat A_i|=2G\sqrt{\hat p(1-\hat p)}\quad\text{(z-score)}.
$$

Dividing by $s$ multiplies a group's weight by $1/\sqrt{\hat p(1-\hat p)}$, which is largest when
$\hat p$ is near 0 or 1. Prompts that are almost always failed or almost always solved count more,
relative to prompts solved half of the time. This is a weighting of prompts by their difficulty under
the current policy.

### Prediction

## Exercise 4 · advantages without the std (Dr. GRPO)

Write the centred advantage: reward minus its group mean, with no division. `R` is `[P, G]`. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_centered_advantages(R):
    return ...          # R minus the mean of its own row; no std

In [ ]:
ex4_ok = s4.check_centered_advantages(globals().get("my_centered_advantages"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def centered_advantages_ref(R):
    return R - R.mean(-1, keepdim=True)

s4.check_centered_advantages(centered_advantages_ref, strict=True)
centered_advantages = mine_or_ref(globals(), "my_centered_advantages", ex4_ok, centered_advantages_ref)

G = 8
ks = torch.arange(0, G + 1)
groups = (torch.arange(G)[None] < ks[:, None]).double()            # row k has k successes
mass_z = group_advantages(groups).abs().sum(-1)
mass_c = centered_advantages(groups).abs().sum(-1)
for k, mz, mc in zip(ks.tolist(), mass_z.tolist(), mass_c.tolist()):
    print(f"k = {k}/8  Σ|Â| z-score {mz:5.2f} | centred {mc:5.2f} | ratio {mz / mc if mc else float('nan'):5.2f}")
print(f"weight of a k = 1 group relative to a k = 4 group: z-score {mass_z[1] / mass_z[4]:.3f}, "
      f"centred {mass_c[1] / mass_c[4]:.3f}")

### Reference explanation

### Reference answer and mechanism

With the z-score a group with one success in eight ($\hat p=1/8$) has total weight $5.29$ against
$8.0$ for a half-solved group: ratio $0.66$. Centred, the totals are $1.75$ and $4.0$: ratio $0.44$.
So the std division raises the relative weight of nearly-failed (and, symmetrically, nearly-solved)
prompts by a factor of $0.66/0.44=1.5$ compared with a half-solved one. Both versions give zero for
$k=0$ and $k=G$. Dr. GRPO calls this weighting a difficulty bias and removes it; the original GRPO
and DAPO keep it. Either way it's a choice of objective, and the update direction across prompts
changes with it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.7))
names = list(weights)
x = np.arange(3)
labels = {names[0]: ["1/15", "1/60", "1/240"], names[1]: ["1/105"] * 3, names[2]: ["1/240"] * 3}
for j, (name, c) in enumerate(zip(names, [BLUE, ORANGE, GREEN])):
    bars = axes[0].bar(x + (j - 1) * 0.26, weights[name].numpy(), width=0.26, color=c, label=name)
    axes[0].bar_label(bars, labels=labels[name], fontsize=7, padding=2)
axes[0].set_yscale("log"); axes[0].set_ylim(1e-3, 0.2)
axes[0].set(xticks=x, xticklabels=["5 tokens", "20 tokens", "80 tokens"], ylabel="per-token weight ∂L/∂ℓ",
            title="Per-token weight, three answers in one batch")
axes[0].legend(fontsize=8, frameon=False)
phat = ks.numpy() / G
axes[1].plot(phat, (mass_z / mass_z[G // 2]).numpy(), "o-", color=BLUE, label="z-score (GRPO)")
axes[1].plot(phat, (mass_c / mass_c[G // 2]).numpy(), "s-", color=ORANGE, label="centred only (Dr. GRPO)")
axes[1].set(xlabel="fraction of the group that succeeded, k/G (G = 8)",
            ylabel="Σ|advantage|, relative to k = 4", title="How much each group counts")
axes[1].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Per-token weight under three loss normalisations, and group weight with and without the std division](../figures/chapter-13/study-path-s4_grpo-02.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part C · Estimating the KL to the reference from sampled tokens

GRPO, like PPO, penalises moving away from a frozen **reference policy** $\pi_{\text{ref}}$ (the model
before RL). At one state the KL divergence is a sum over the whole vocabulary (D35):

$$
\mathrm{KL}(\pi\,\|\,\pi_{\text{ref}})=\sum_v \pi(v)\log\frac{\pi(v)}{\pi_{\text{ref}}(v)}.
$$

D35's example has three tokens, $\pi=[0.7,0.2,0.1]$ and $\pi_{\text{ref}}=[0.5,0.3,0.2]$, and
KL $=0.0851$. Qwen3 has 151,936 tokens, and training code usually keeps only the log-probability
of the token that was actually sampled. With $y\sim\pi$ and $d=\log\pi_{\text{ref}}(y)-\log\pi(y)$,
two per-token estimators are common ([Schulman, 2020](http://joschu.net/blog/kl-approx.html)):

$$
k_1(y)=\log\pi(y)-\log\pi_{\text{ref}}(y)=-d,\qquad k_3(y)=e^{d}-d-1.
$$

Both are **unbiased**: their mean over $y\sim\pi$ is the KL. For $k_1$ that's the definition. For
$k_3$, $\mathbb E_\pi[e^d]=\sum_v\pi(v)\frac{\pi_{\text{ref}}(v)}{\pi(v)}=1$, so
$\mathbb E_\pi[k_3]=1+\mathrm{KL}-1=\mathrm{KL}$. A single $k_1$ value can be negative; $k_3$ never
is, because $e^d\ge1+d$ for every real $d$. PPO in Day 15 · 04 put $k_1$ into the per-token reward;
GRPO adds $\beta\cdot k_3$ to the loss (D35).

### Prediction

## Exercise 5 · k1 and k3

Write both estimators from the log-probabilities of the sampled tokens (any shape, elementwise).
One line for `k1`, two for `k3`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_k1(logp, ref_logp):
    return ...          # log π − log π_ref

def my_k3(logp, ref_logp):
    d = ...             # log π_ref − log π
    return ...          # e^d − d − 1

In [ ]:
ex5_ok = s4.check_k1_k3(globals().get("my_k1"), globals().get("my_k3"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def k1_ref(logp, ref_logp):
    return logp - ref_logp

def k3_ref(logp, ref_logp):
    d = ref_logp - logp
    return torch.exp(d) - d - 1

s4.check_k1_k3(k1_ref, k3_ref, strict=True)
k1 = mine_or_ref(globals(), "my_k1", ex5_ok, k1_ref)
k3 = mine_or_ref(globals(), "my_k3", ex5_ok, k3_ref)

pi, ref_pi = torch.tensor(s4.D35_PI, dtype=torch.float64), torch.tensor(s4.D35_REF, dtype=torch.float64)
print("token        A       B       C")
print("π         ", "  ".join(f"{v:.3f}" for v in pi.tolist()))
print("π_ref     ", "  ".join(f"{v:.3f}" for v in ref_pi.tolist()))
print("k1        ", "  ".join(f"{v:+.3f}" for v in k1(pi.log(), ref_pi.log()).tolist()))
print("k3        ", "  ".join(f"{v:.3f}" for v in k3(pi.log(), ref_pi.log()).tolist()))
print(f"E_π[k1] = {(pi * k1(pi.log(), ref_pi.log())).sum():.4f}   E_π[k3] = {(pi * k3(pi.log(), ref_pi.log())).sum():.4f}")

### Reference explanation

### Reference answer and mechanism

Token A is more likely under $\pi$ than under $\pi_{\text{ref}}$, so $k_1>0$; tokens B and C are less
likely, so their $k_1$ is negative. $k_3$ is $0.051, 0.095, 0.307$, all positive, and both
expectations under $\pi$ equal the exact KL $0.0851$ (D35). Swapping the sign of $d$ gives
$e^{-d}+d-1$, which is the matching estimator for the other direction, $\mathrm{KL}(\pi_{\text{ref}}\|\pi)$,
only when tokens are sampled from $\pi_{\text{ref}}$; here it would be biased.

### Prediction

## Exercise 6 · the exact KL

Write the exact KL between two distributions over the last dimension, so the sampled estimators have
something to be compared with. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_kl_exact(p, q):
    return ...          # Σ_v p_v (log p_v − log q_v), over the last dimension

In [ ]:
ex6_ok = s4.check_kl_exact(globals().get("my_kl_exact"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def kl_exact_ref(p, q):
    return (p * (p.log() - q.log())).sum(-1)

s4.check_kl_exact(kl_exact_ref, strict=True)
kl_exact = mine_or_ref(globals(), "my_kl_exact", ex6_ok, kl_exact_ref)
print(f"KL(π‖π_ref) = {kl_exact(pi, ref_pi):.4f}   KL(π_ref‖π) = {kl_exact(ref_pi, pi):.4f}  (not symmetric)")

### Reference explanation

### Reference answer and mechanism

$\mathrm{KL}(\pi\|\pi_{\text{ref}})=0.0851$ and $\mathrm{KL}(\pi_{\text{ref}}\|\pi)=0.0920$: the KL is
not symmetric, so it matters which distribution weights the log-ratios. In RL fine-tuning the
penalty is $\mathrm{KL}(\pi_\theta\|\pi_{\text{ref}})$, weighted by the policy being trained, because
the tokens come from that policy.

### Prediction

## Predict 3 · which estimator is less noisy?

Sample one token $y\sim\pi$ and compute $k_1(y)$ and $k_3(y)$. On the D35 example, which estimator
has the smaller standard deviation? Then take a policy that has moved far from the reference:
$\pi=[0.98,0.015,0.005]$, $\pi_{\text{ref}}=[0.2,0.3,0.5]$. Which one is smaller now, and why?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
CASES = {"near (D35)": ([0.7, 0.2, 0.1], [0.5, 0.3, 0.2]), "far": ([0.98, 0.015, 0.005], [0.2, 0.3, 0.5])}
for name, (p_, q_) in CASES.items():
    p_, q_ = torch.tensor(p_, dtype=torch.float64), torch.tensor(q_, dtype=torch.float64)
    for est_name, est in [("k1", k1), ("k3", k3)]:
        vals = est(p_.log(), q_.log())                                # value for each possible token
        mean = (p_ * vals).sum()                                      # exact expectation over y ~ π
        sd = (p_ * (vals - mean) ** 2).sum().sqrt()
        print(f"{name:11s} {est_name}: values {[round(v, 3) for v in vals.tolist()]}  mean {mean:.4f}  sd {sd:.4f}")
    print(f"{'':11s} exact KL = {kl_exact(p_, q_):.4f}")

### Reference explanation

### Reference answer and mechanism

These are exact moments, computed by enumerating the three tokens. **Near the reference** (D35),
$k_3$ has standard deviation $0.076$ against $0.391$ for $k_1$: about 5 times smaller, because
$e^d-d-1\approx d^2/2$ is small when $d$ is small, while $k_1=-d$ swings between signs.

**Far from the reference** the order reverses: $k_3$ has standard deviation $6.84$ against $0.71$ for
$k_1$, ten times larger. Token C has probability 0.5% under $\pi$ but 50% under $\pi_{\text{ref}}$, so
on the rare draws of C, $d=\log(0.5/0.005)=4.61$ and $k_3=100-4.61-1=94.4$. Both estimators still
average to the same KL, 1.4895. The low-variance property of $k_3$ holds near the reference, not in general
(Chapter 13 §13.4).

The histograms below show what this does to an average over 16 sampled tokens. Far from the reference,
72% of such batches ($0.98^{16}$) contain neither B nor C, and their $k_3$ average is $0.79$, about
half the true KL; the expectation is restored by rare batches with very large values (note the
logarithmic count axis).

In [ ]:
gen = torch.Generator().manual_seed(3)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.7))
for ax, (name, (p_, q_)) in zip(axes, CASES.items()):
    p_, q_ = torch.tensor(p_, dtype=torch.float64), torch.tensor(q_, dtype=torch.float64)
    ys = torch.multinomial(p_, 5000 * 16, replacement=True, generator=gen).view(5000, 16)
    lp, lq = p_.log()[ys], q_.log()[ys]
    m1, m3 = k1(lp, lq).mean(1), k3(lp, lq).mean(1)                    # 5,000 averages of 16 sampled tokens
    lo, hi = float(min(m1.min(), m3.min())), float(max(m1.max(), m3.max()))
    bins = np.linspace(lo, hi, 60 if name.startswith("near") else 150)
    if not name.startswith("near"):
        ax.set_yscale("log")                                         # the k3 tail is rare but large
    ax.hist(m1.numpy(), bins=bins, color=BLUE, alpha=0.55, label=f"k1  (sd {m1.std():.3f})")
    ax.hist(m3.numpy(), bins=bins, color=ORANGE, alpha=0.55, label=f"k3  (sd {m3.std():.3f})")
    ax.axvline(kl_exact(p_, q_).item(), color="black", ls="--", lw=1, label=f"exact KL {kl_exact(p_, q_):.3f}")
    ax.set(xlabel="mean of 16 sampled-token estimates", ylabel="count (5,000 batches)", title=f"{name}")
    ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Distributions of 16-token averages of the k1 and k3 KL estimators, near and far from the reference](../figures/chapter-13/study-path-s4_grpo-03.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

An estimator with the right **value** doesn't automatically give the right **gradient**. In GRPO the
sampled token $y$ is fixed data and $\beta\,k_3(y)$ is differentiated through $\log\pi_\theta(y)$. With
$d=\log\pi_{\text{ref}}(y)-\log\pi_\theta(y)$,

$$
\frac{\partial k_3}{\partial\log\pi_\theta(y)}=1-e^{d}=1-\frac{\pi_{\text{ref}}(y)}{\pi_\theta(y)} .
$$

Averaging over $y\sim\pi_\theta$ with the samples held fixed gives
$\sum_y\pi_\theta(y)\bigl(1-\tfrac{\pi_{\text{ref}}(y)}{\pi_\theta(y)}\bigr)\nabla\log\pi_\theta(y)
=\sum_y\nabla\pi_\theta(y)-\sum_y\pi_{\text{ref}}(y)\nabla\log\pi_\theta(y)=-\sum_y\pi_{\text{ref}}(y)\nabla\log\pi_\theta(y)$,
which is $\nabla_\theta\,\mathrm{KL}(\pi_{\text{ref}}\|\pi_\theta)$: the gradient of the KL in the
**other** direction. With respect to the logits $z$ of a softmax it is simply $\pi_\theta-\pi_{\text{ref}}$.
The next exercise checks this with autograd.

### Prediction

## Exercise 7 · what gradient does k3 give in the loss?

Build the expected k3 loss term at one state with the sampled token held fixed:
$\sum_v p_v\,k_3(v)$ where the weights $p_v=\pi_\theta(v)$ are **detached** (they are the sampling
probabilities, not something the loss differentiates through), while $k_3$ is computed from
`logp = z.log_softmax(-1)`, which carries the gradient. One line, using `k3` from Exercise 5.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_k3_surrogate(z, ref_logp):
    logp = z.log_softmax(-1)
    return ...          # Σ_v p_v · k3(logp_v, ref_logp_v), with p = logp.exp() detached

In [ ]:
ex7_ok = s4.check_k3_sampled_surrogate(globals().get("my_k3_surrogate"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def k3_surrogate_ref(z, ref_logp):
    logp = z.log_softmax(-1)
    return (logp.exp().detach() * k3(logp, ref_logp)).sum()

s4.check_k3_sampled_surrogate(k3_surrogate_ref, strict=True)
k3_surrogate = mine_or_ref(globals(), "my_k3_surrogate", ex7_ok, k3_surrogate_ref)

def grad_wrt_logits(f, p_):
    z = p_.log().clone().requires_grad_(True)
    (g,) = torch.autograd.grad(f(z), z)
    return g

for name, (p_, q_) in CASES.items():
    p_, q_ = torch.tensor(p_, dtype=torch.float64), torch.tensor(q_, dtype=torch.float64)
    g_k3 = grad_wrt_logits(lambda z: k3_surrogate(z, q_.log()), p_)
    g_k1 = grad_wrt_logits(lambda z: (z.log_softmax(-1).exp().detach() * k1(z.log_softmax(-1), q_.log())).sum(), p_)
    g_fwd = grad_wrt_logits(lambda z: kl_exact(z.softmax(-1), q_), p_)       # ∇ KL(π‖π_ref)
    g_rev = grad_wrt_logits(lambda z: kl_exact(q_, z.softmax(-1)), p_)       # ∇ KL(π_ref‖π) = π − π_ref
    print(f"{name}:")
    for label, g in [("k3 in the loss", g_k3), ("∇ KL(π_ref‖π)", g_rev), ("∇ KL(π‖π_ref)", g_fwd), ("k1 in the loss", g_k1)]:
        print(f"   {label:15s} {[round(v, 4) for v in g.tolist()]}")
    cos = torch.nn.functional.cosine_similarity(g_k3, g_fwd, dim=0)
    print(f"   cosine(k3 gradient, ∇ KL(π‖π_ref)) = {cos:.3f}, norm ratio {g_k3.norm() / g_fwd.norm():.2f}")

### Reference explanation

### Reference answer and mechanism

The $k_3$ term's expected gradient equals $\pi-\pi_{\text{ref}}$ exactly, the gradient of
$\mathrm{KL}(\pi_{\text{ref}}\|\pi)$. Near the reference (D35) it is close to the gradient of the
KL we meant, $\mathrm{KL}(\pi\|\pi_{\text{ref}})$: cosine 0.998, norm ratio 1.13, because both
divergences agree to second order around $\pi=\pi_{\text{ref}}$. Far from the reference they
differ: in the far example the $k_3$ gradient is 7.9 times larger than the forward-KL gradient, at
cosine 0.93. So "GRPO adds $\beta\cdot k_3$" is a penalty whose value estimates one KL and whose
gradient follows the other. Both pull $\pi$ towards $\pi_{\text{ref}}$; the $k_3$ gradient pulls much
harder on a policy that has nearly abandoned tokens the reference likes.

The last row shows why $k_1$ is not added to the loss this way: its expected gradient is exactly
zero, since $\sum_y\pi(y)\nabla\log\pi(y)=\nabla\sum_y\pi(y)=0$. PPO puts $k_1$ into the
**reward** instead, where it multiplies $\nabla\log\pi$ through the advantage (D35, S3).

---

# Part D · The GRPO loss

Now the pieces go together. For answer $i$ and token $t$, the **importance ratio** is
$\rho_{it}=\exp\bigl(\log\pi_\theta(y_{it}\mid s_{it})-\log\pi_{\text{old}}(y_{it}\mid s_{it})\bigr)$,
where $\pi_{\text{old}}$ is the policy that sampled the batch. The original GRPO objective, with the
per-answer mean of Part B and $k_3$ of Part C, is

$$
J(\theta)=\frac1N\sum_{i=1}^N\frac1{|o_i|}\sum_{t=1}^{|o_i|}\Bigl[\min\bigl(\rho_{it}\hat A_i,\;
\mathrm{clip}(\rho_{it},1-\epsilon,1+\epsilon)\hat A_i\bigr)-\beta\,k_{3,it}\Bigr],\qquad L=-J .
$$

Compared with PPO in S3 three things changed: $\hat A_i$ is one number per answer (no value model, no
GAE), the KL is a term of the loss rather than a per-token reward, and the advantage is not whitened
over the batch. We use $\epsilon=0.2$ and $\beta=0.04$ (the $\beta$ of DeepSeekMath and of Day 15 · 04).

### Prediction

## Exercise 8 · the GRPO loss

Write the per-token loss. `logp`, `old_logp`, `ref_logp` and `mask` are `[N, T]`, `A` is `[N]`.
Three lines: the ratio from log-probabilities, the clipped surrogate with `A` broadcast over tokens
(`A[:, None]`), and the per-token loss term $-\text{surr}+\beta\,k_3$. The aggregation (`seq_mean`
from Exercise 3) is given.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_grpo_loss(logp, old_logp, ref_logp, A, mask, eps=0.2, beta=0.04):
    ratio = ...         # π_θ / π_old per token; old_logp is data (no gradient)
    surr = ...          # min(ρ·A, clip(ρ, 1−ε, 1+ε)·A), with A[:, None]
    per_token = ...     # −surr + β·k3(logp, ref_logp)
    return seq_mean(per_token, mask)

In [ ]:
ex8_ok = s4.check_grpo_loss(globals().get("my_grpo_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def grpo_loss_ref(logp, old_logp, ref_logp, A, mask, eps=0.2, beta=0.04):
    ratio = torch.exp(logp - old_logp.detach())
    surr = torch.minimum(ratio * A[:, None], ratio.clamp(1 - eps, 1 + eps) * A[:, None])
    per_token = -surr + beta * k3(logp, ref_logp)
    return seq_mean(per_token, mask)

s4.check_grpo_loss(grpo_loss_ref, strict=True)
grpo_loss = mine_or_ref(globals(), "my_grpo_loss", ex8_ok, grpo_loss_ref)

### Reference explanation

### Reference answer and mechanism

`old_logp` is recorded when the batch is sampled and is treated as data. At a fresh batch
$\log\pi_\theta=\log\pi_{\text{old}}$, so every $\rho=1$ and clipping is inactive, yet
$\partial\rho/\partial\log\pi_\theta=\rho=1$: the surrogate's gradient is $-\hat A_i/(N|o_i|)$ per
token, the policy gradient with the group baseline. The KL term adds
$\beta(1-\pi_{\text{ref}}/\pi_\theta)/(N|o_i|)$.

`torch.minimum` takes the pessimistic branch. For $\hat A>0$ the clipped branch is the smaller one once
$\rho>1+\epsilon$, and it is constant in $\theta$, so that token stops contributing gradient; for
$\hat A<0$ the same happens below $1-\epsilon$. On the harmful side ($\hat A>0$, $\rho<1$) the
unclipped term stays and the gradient keeps pushing the token back (S3, Chapter 13 §13.3).

To run GRPO end to end in seconds we need a policy that is small enough to inspect. The toy below
has **16 tasks** (prompts). An answer is **3 tokens** from a vocabulary of **5**. The policy is a table
of logits `theta[task, position, previous token, next token]`, shape `[16, 3, 6, 5]` (previous
token 5 means "start"), and

$$
\pi_\theta(y\mid\text{task})=\prod_{t=1}^{3}\mathrm{softmax}\bigl(\theta[\text{task},t,y_{t-1}]\bigr)[y_t].
$$

The reward is 1 if the answer equals the task's target string and 0 otherwise. The initial table is
$0.5\times$ Gaussian noise plus a bias towards the target that grows linearly from 0 (task 0) to 2.5
(task 15), so tasks range from hard (success probability ≈ 0.01) to easy (≈ 0.4). Because the target
is a single string, its exact success probability is one product of table entries: no sampling
noise in the evaluation.

In [ ]:
TASKS, L_TOY, V_TOY, G_TOY = 16, 3, 5, 8
TARGET = torch.randint(0, V_TOY, (TASKS, L_TOY), generator=torch.Generator().manual_seed(7))
BIAS = torch.linspace(0.0, 2.5, TASKS)
toy_tasks = torch.arange(TASKS).repeat_interleave(G_TOY)        # rows grouped by task: t0 ×8, t1 ×8, …
toy_mask = torch.ones(TASKS * G_TOY, L_TOY, dtype=torch.float64)  # every toy answer has 3 tokens

def toy_init(seed):
    g = torch.Generator().manual_seed(seed)
    theta = 0.5 * torch.randn(TASKS, L_TOY, V_TOY + 1, V_TOY, generator=g, dtype=torch.float64)
    theta += BIAS[:, None, None, None] * torch.nn.functional.one_hot(TARGET, V_TOY)[:, :, None, :]
    return theta.requires_grad_()

@torch.no_grad()
def toy_sample(theta, tasks, gen):
    prev, ys = torch.full((len(tasks),), V_TOY), []
    for t in range(L_TOY):
        y = torch.multinomial(theta[tasks, t, prev].softmax(-1), 1, generator=gen).squeeze(-1)
        ys.append(y); prev = y
    return torch.stack(ys, 1)                                    # [N, 3]

def toy_logp(theta, tasks, ys):
    prev = torch.cat([torch.full((len(tasks), 1), V_TOY), ys[:, :-1]], 1)
    logits = theta[tasks[:, None], torch.arange(L_TOY)[None], prev]       # [N, 3, V]
    return logits.log_softmax(-1).gather(-1, ys[..., None]).squeeze(-1)  # [N, 3] = log π(y_t | s_t)

def toy_reward(ys, tasks):
    return (ys == TARGET[tasks]).all(1).double()

def toy_success(theta):                                          # exact P(answer = target), per task
    with torch.no_grad():
        return toy_logp(theta, torch.arange(TASKS), TARGET).sum(1).exp()

theta0 = toy_init(0)
print("exact initial success probability per task:", [round(v, 3) for v in toy_success(theta0).tolist()])
print(f"mean over tasks {toy_success(theta0).mean():.3f}")

### Prediction

## Predict 4 · one update or several on the same batch

Sample one batch from the toy policy (16 tasks × 8 answers), record `old_logp`, and take **8 Adam
steps on that same batch** with the GRPO loss ($\beta=0$, learning rate 0.05). Before step 1, what is
the range of the ratios $\rho$, what fraction of tokens is clipped, and what is the loss value? How do
those numbers change by step 8? Does clipping keep every ratio inside $[0.8, 1.2]$?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
theta = toy_init(0)
gen = torch.Generator().manual_seed(1)
ys = toy_sample(theta, toy_tasks, gen)
R = toy_reward(ys, toy_tasks)
A = group_advantages(R.view(TASKS, G_TOY)).flatten()
old_logp = toy_logp(theta, toy_tasks, ys).detach()
opt = torch.optim.Adam([theta], lr=0.05)
moving = (A != 0)[:, None].expand(-1, L_TOY)                      # tokens of answers with a nonzero advantage
print(f"{(A != 0).sum()} of {len(A)} answers have Â ≠ 0 ({int((R.view(TASKS, G_TOY).std(-1) > 0).sum())} mixed groups)")
for step in range(1, 9):
    logp = toy_logp(theta, toy_tasks, ys)
    loss = grpo_loss(logp, old_logp, logp.detach(), A, toy_mask, 0.2, 0.0)
    ratio = (logp - old_logp).exp().detach()
    clipped = ((ratio > 1.2) & (A[:, None] > 0)) | ((ratio < 0.8) & (A[:, None] < 0))
    if step == 1:
        assert torch.equal(ratio, torch.ones_like(ratio)) and abs(loss.item()) < 1e-12
    print(f"step {step}: ρ in [{ratio[moving].min():.3f}, {ratio[moving].max():.3f}]  "
          f"clipped {clipped[moving].float().mean():.3f}  loss {loss.item():+.4f}".replace("-0.0000", " 0.0000"))
    opt.zero_grad(); loss.backward(); opt.step()

### Reference explanation

### Reference answer and mechanism

Before step 1 the batch is fresh: every $\rho$ is exactly 1, nothing is clipped, and the loss is
exactly 0, because within each group the advantages sum to zero and all answers have 3 tokens. The
gradient is not zero: it is the group-baseline policy gradient. With one update per batch, as in
Day 15 · 04 Part F and in Part F below, the clip never acts.

Reusing the batch makes the ratios drift apart: by step 8 they span about $[0.59, 1.56]$ and about
60% of the tokens with $\hat A\neq0$ sit on the clipped side. Clipping does **not** keep the ratios
inside $[0.8,1.2]$. It removes a token's own gradient once it has moved far enough in the helpful
direction, but the token's probability keeps changing: other answers in the batch share the same
table rows, and Adam's momentum carries earlier steps forward. The clip bounds the surrogate, not the
policy (Chapter 13 §13.3).

---

# Part E · Toy GRPO against REINFORCE with no baseline

S3 built REINFORCE: weight each token's $\nabla\log\pi$ by the raw reward $R_i$, with no baseline.
GRPO weights it by $\hat A_i$. With a 0/1 reward the two look close: a failed answer has $R=0$ and
contributes nothing to REINFORCE, while GRPO pushes failed answers down whenever their group is mixed.
We compare four runs on the toy, each with Adam (learning rate 0.05), one update per batch, 100
iterations, all 16 tasks × 8 answers per batch, and 8 seeds:

- **GRPO** with the reward $R\in\{0,1\}$, and **GRPO with $R+1$** (rewards $\{1,2\}$, same ranking)
- **REINFORCE** with $R\in\{0,1\}$, and **REINFORCE with $R+1$**

The measured quantity is the exact success probability averaged over the 16 tasks.

### Prediction

## Exercise 9 · REINFORCE with no baseline

Write REINFORCE's loss in the same per-token form: the term for each token is $-R_i\log\pi_\theta(y_{it})$,
aggregated with `seq_mean`. `R` is `[N]`. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_reinforce_loss(logp, R, mask):
    return ...          # seq_mean of −R_i · log π per token (R[:, None]); no baseline

In [ ]:
ex9_ok = s4.check_reinforce_loss(globals().get("my_reinforce_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def reinforce_loss_ref(logp, R, mask):
    return seq_mean(-R[:, None] * logp, mask)

s4.check_reinforce_loss(reinforce_loss_ref, strict=True)
reinforce_loss = mine_or_ref(globals(), "my_reinforce_loss", ex9_ok, reinforce_loss_ref)

def toy_run(method, seed, shift=0.0, iters=100, lr=0.05):
    theta = toy_init(seed)
    opt = torch.optim.Adam([theta], lr=lr)
    gen = torch.Generator().manual_seed(1000 + seed)
    success, zero_var = [toy_success(theta).mean().item()], []
    for _ in range(iters):
        ys = toy_sample(theta, toy_tasks, gen)
        R = toy_reward(ys, toy_tasks) + shift
        logp = toy_logp(theta, toy_tasks, ys)
        if method == "grpo":
            A = group_advantages(R.view(TASKS, G_TOY)).flatten()
            loss = grpo_loss(logp, logp.detach(), logp.detach(), A, toy_mask, 0.2, 0.0)   # β = 0 in the toy
        else:
            loss = reinforce_loss(logp, R, toy_mask)
        opt.zero_grad(); loss.backward(); opt.step()
        success.append(toy_success(theta).mean().item())
        zero_var.append((R.view(TASKS, G_TOY).std(-1) == 0).float().mean().item())
    return np.array(success), np.array(zero_var)

SEEDS = range(8)
t0 = time.time()
toy = {}
for label, method, shift in [("GRPO", "grpo", 0.0), ("GRPO, R+1", "grpo", 1.0),
                             ("REINFORCE", "reinforce", 0.0), ("REINFORCE, R+1", "reinforce", 1.0)]:
    runs = [toy_run(method, s, shift) for s in SEEDS]
    toy[label] = (np.stack([r[0] for r in runs]), np.stack([r[1] for r in runs]))
gap = np.abs(toy["GRPO"][0] - toy["GRPO, R+1"][0]).max()
assert gap < 1e-9                                                # same run up to floating-point rounding
print(f"largest |GRPO − GRPO(R+1)| over all seeds and iterations: {gap:.1e}")
se = lambda x: x.std(ddof=1) / math.sqrt(len(x))
for label, (succ, zv) in toy.items():
    print(f"{label:15s} success @30 {succ[:, 30].mean():.3f} ± {se(succ[:, 30]):.3f}   "
          f"@100 {succ[:, 100].mean():.3f} ± {se(succ[:, 100]):.3f}   "
          f"zero-variance groups: first 10 its {zv[:, :10].mean():.2f}, last 10 {zv[:, -10:].mean():.2f}")
for it in (30, 100):
    d = toy["REINFORCE"][0][:, it] - toy["GRPO"][0][:, it]           # paired by seed (same initial table)
    print(f"REINFORCE − GRPO at iteration {it}: {d.mean():+.3f} ± {se(d):.3f}")
print(f"({time.time() - t0:.1f} s; ± is the standard error over {len(SEEDS)} seeds)")

### Reference explanation

### Reference answer and mechanism

At $\rho=1$ REINFORCE's gradient is $-R_i/(N|o_i|)$ per token, GRPO's is $-\hat A_i/(N|o_i|)$:
the only difference is the weight. With $R\in\{0,1\}$, the answer with $R=0$ gets no gradient at
all in REINFORCE.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
its = np.arange(101)
styles = {"GRPO": (BLUE, "-"), "GRPO, R+1": (BLUE, ":"), "REINFORCE": (ORANGE, "-"), "REINFORCE, R+1": (RED, "-")}
for label, (succ, zv) in toy.items():
    c, ls = styles[label]
    mean, err = succ.mean(0), succ.std(0, ddof=1) / math.sqrt(len(succ))
    axes[0].plot(its, mean, color=c, ls=ls, lw=2.2 if ls == ":" else 1.5, label=label)
    axes[0].fill_between(its, mean - 2 * err, mean + 2 * err, color=c, alpha=0.15)
axes[0].set(xlabel="iteration (one update per batch of 128 answers)", ylabel="exact success probability (mean of 16 tasks)",
            title="Toy policy: 8 seeds, band = ±2 SE")
axes[0].legend(fontsize=8, frameon=False)
zv = toy["GRPO"][1]
axes[1].plot(np.arange(1, 101), zv.mean(0), color=BLUE, label="GRPO, measured")
axes[1].fill_between(np.arange(1, 101), zv.mean(0) - 2 * zv.std(0, ddof=1) / math.sqrt(len(zv)),
                     zv.mean(0) + 2 * zv.std(0, ddof=1) / math.sqrt(len(zv)), color=BLUE, alpha=0.15)
axes[1].set(xlabel="iteration", ylabel="fraction of the 16 groups", ylim=(0, 1),
            title="Zero-variance groups in each GRPO batch")
axes[1].legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Toy GRPO vs REINFORCE learning curves over 8 seeds, and the fraction of zero-variance groups in GRPO](../figures/chapter-13/study-path-s4_grpo-04.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

**Reading the toy results.** Three things are exact or well outside the noise:

1. **GRPO is invariant to the reward offset.** GRPO with $R$ and with $R+1$ produce the same run up
   to floating-point rounding (largest difference about $10^{-14}$, asserted above): the group mean
   removes the offset and the std removes the scale.
2. **REINFORCE is not.** With $R+1$ every sampled answer, wrong ones included, gets a positive weight
   and is pushed up; only the difference between weights 2 and 1 carries information. It reaches a
   success probability of 0.65 at iteration 100, against 0.95–0.97 for the other three.
   That is the variance a baseline removes (S3, D39).
3. **With a 0/1 reward, no baseline was slightly better here.** REINFORCE with $R\in\{0,1\}$ is
   *ahead* of GRPO by $+0.042\pm0.002$ at iteration 30 and $+0.014\pm0.001$ at iteration 100
   (differences paired by seed, so each pair starts from the same table). One difference between the update rules that can explain this: once a task is solved by
   all 8 answers, GRPO's group gives zero advantage, while REINFORCE still pushes those correct answers
   up. In this toy, raising an already-correct answer never hurts. The right panel shows how fast
   zero-variance groups take over: about 40% of GRPO's groups give no signal in the first 10
   iterations (the hard tasks fail 8 out of 8 times), and about 75% by the end (most tasks are solved
   8 out of 8 times).

So the baseline's job is to remove the reward offset, and GRPO's group baseline does that exactly.
It doesn't make learning faster by itself on a 0/1 reward with this optimizer. This toy has no
shared parameters across tasks, uses Adam (which rescales each parameter's gradient) and no KL term;
the comparison doesn't carry over to a language model as a ranking.

---

# Part F · GRPO on Qwen3-0.6B

Now the same update on a real model. Two networks are loaded: the **policy** (trained) and a frozen
copy of it as $\pi_{\text{ref}}$. There is **no value model** and **no reward network**: the reward
is a rule computed from the answer text.

**Why this reward.** It has to be computable by a program and movable by a 0.6B model in a few dozen
updates. Correctness tasks with a verifier are S5's subject. Here each HH-RLHF question gets the
suffix *"Answer in one or two sentences."*, and the reward is

$$
R=1\quad\text{if the answer stops by itself (emits \texttt{<|im\_end|>} within 48 tokens) and has 5–20 words; else }0.
$$

In a probe on the untrained model, about 27% of answers satisfied it. At $p\approx0.27$, Part A gives
$P(\text{zero variance})=0.73^8+0.27^8\approx0.08$ for $G=8$, so most groups are mixed and carry a
signal. The rule says nothing about whether an answer is correct or helpful: this is a test of the
mechanism, and we'll look at what the model writes to satisfy it.

**Set-up.** 40 iterations; each samples 4 new training questions × $G=8$ answers at temperature 1
(so $k_3$ is unbiased), computes rewards and advantages, and takes **one** AdamW step (lr $10^{-6}$,
gradient norm clipped to 1). One step per batch means $\rho=1$ and the clip is inactive (Part D). The
policy keeps float32 weights and runs its forward passes in bf16 autocast; with bf16 weights, most
AdamW updates of size $10^{-6}$ would be rounded away. The reference is float32 under the same
autocast, so at iteration 1 $\pi_\theta=\pi_{\text{ref}}$ exactly and $k_3=0$.

First, the two shared helpers that sample answers and compute per-token log-probabilities.

In [ ]:
print(inspect.getsource(Q.rollout))
print(inspect.getsource(Q.response_texts))
print(inspect.getsource(Q.token_logps))

`rollout` samples from the full distribution (temperature 1, no top-k/top-p) and returns the token
ids, a response mask that is 1 through the first `<|im_end|>`, and a `truncated` flag for answers that
hit the token budget. `response_texts` decodes each answer up to and including its first
`<|im_end|>` (special tokens dropped). `token_logps` runs one forward pass and reads, at response position $t$, the
logit at prompt length $P-1+t$ for the token at $P+t$: the shift by one between a logit and the token
it predicts.

In [ ]:
tok = Q.load_tokenizer()
SUFFIX = " Answer in one or two sentences."
train_q = Q.hh_questions("train", start=20000)
test_q = Q.hh_questions("test")[:96]
print(f"{len(train_q)} training questions available, {len(test_q)} held-out test questions used")
print("example:", train_q[0] + SUFFIX)

def ask(q):
    return [{"role": "user", "content": q + SUFFIX}]

### Prediction

## Exercise 10 · the length-target reward

Write the rule: 1.0 if the answer was not truncated and has between 5 and 20 words (bounds
included), else 0.0. Count words with `text.split()`. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_length_reward(text, truncated):
    n = ...             # number of words
    return ...          # 1.0 if not truncated and 5 ≤ n ≤ 20, else 0.0

In [ ]:
ex10_ok = s4.check_length_reward(globals().get("my_length_reward"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def length_reward_ref(text, truncated):
    n = len(text.split())
    return float((not truncated) and 5 <= n <= 20)

s4.check_length_reward(length_reward_ref, strict=True)
length_reward = mine_or_ref(globals(), "my_length_reward", ex10_ok, length_reward_ref)

def rewards_of(ro):
    texts = Q.response_texts(tok, ro)
    R = torch.tensor([length_reward(t, bool(tr)) for t, tr in zip(texts, ro["truncated"].tolist())])
    return R, texts

### Reference explanation

### Reference answer and mechanism

The `truncated` flag matters: an answer cut off at 48 tokens may still have fewer than 20 words if it
contains long tokens or markup, but it didn't stop by itself, so it gets 0. Word counting with
`split()` treats newlines and list markers as separators, so a bulleted answer counts its words, not
its lines. A rule is only as good as these edge cases; S5 tests a real verifier much more thoroughly.

In [ ]:
autocast = lambda: torch.autocast("cuda", dtype=torch.bfloat16)
MAX_NEW, G, PROMPTS, ITERS, LR, BETA, EPS, CHUNK = 48, 8, 4, 40, 1e-6, 0.04, 0.2, 16
torch.cuda.reset_peak_memory_stats()
policy = Q.load_chat_model(dtype=torch.float32)
ref = Q.load_chat_model(dtype=torch.float32).eval().requires_grad_(False)
n_params = sum(p.numel() for p in policy.parameters())
print(f"policy: {n_params / 1e6:.0f} M parameters (float32), trained | reference: frozen copy | "
      f"GPU memory now {Q.gib(torch.cuda.memory_allocated()):.1f} GiB")

# one group of G = 8 answers to one held-out question, before any training
torch.manual_seed(5)
with autocast():
    ro = Q.rollout(policy, tok, [ask(test_q[3])] * G, max_new=MAX_NEW)
R, texts = rewards_of(ro)
A = group_advantages(R.view(1, G)).flatten()
print("QUESTION:", test_q[3] + SUFFIX)
for i in R.argsort(descending=True).tolist():
    flag = "truncated" if ro["truncated"][i] else f"{len(texts[i].split())} words"
    print(f"R {R[i]:.0f}  Â {A[i]:+.2f}  [{flag:>9s}]  {textwrap.shorten(texts[i], 100)}")

### Prediction

## Exercise 11 · advantages and loss inside the training loop

These are the two lines the training loop below calls for each chunk of the batch. Rows of `R`,
`logp` and `ref_logp` are grouped by prompt: $G$ consecutive answers per question. Compute the group
advantages as a flat `[N]` vector, then the loss for a **fresh batch with one update**, where the
behaviour log-probabilities are the current ones, detached. Use `group_advantages` and `grpo_loss`,
with `EPS` and `BETA`. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_grpo_terms(R, logp, ref_logp, mask, G):
    A = ...             # z-score within each group of G consecutive rows, flattened to [N]
    loss = ...          # grpo_loss(...) with old_logp = logp.detach(), EPS, BETA
    return A, loss

In [ ]:
ex11_ok = s4.check_grpo_terms(globals().get("my_grpo_terms"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def grpo_terms_ref(R, logp, ref_logp, mask, G):
    A = group_advantages(R.view(-1, G)).flatten()
    loss = grpo_loss(logp, logp.detach(), ref_logp, A, mask, EPS, BETA)
    return A, loss

s4.check_grpo_terms(grpo_terms_ref, strict=True)
grpo_terms = mine_or_ref(globals(), "my_grpo_terms", ex11_ok, grpo_terms_ref)

### Reference explanation

### Reference answer and mechanism

`R.view(-1, G)` puts each prompt's $G$ answers in one row, because the rollout lists them
consecutively (`[q0]*8 + [q1]*8 + …`); `view(G, -1)` would mix answers from different prompts into a
group. Passing `logp.detach()` as `old_logp` gives $\rho=\exp(\log\pi-\mathrm{sg}(\log\pi))=1$ with
gradient 1, which is exactly the one-update case and saves a second forward pass. Passing `logp`
itself (not detached) would make $\rho$ a constant and silently remove the policy-gradient term,
leaving only the KL.

In [ ]:
@torch.no_grad()
def evaluate(model, n_samples=2, seed=123):
    """Held-out reward: 96 test questions × n_samples answers, same sampler and reward as training."""
    torch.manual_seed(seed)
    Rs, Ls, texts = [], [], []
    for i in range(0, len(test_q), 48):
        with autocast():
            ro = Q.rollout(model, tok, [ask(q) for q in test_q[i:i + 48] for _ in range(n_samples)], max_new=MAX_NEW)
        R, t = rewards_of(ro)
        Rs.append(R.view(-1, n_samples)); Ls.append(ro["mask"].sum(1).float().cpu().view(-1, n_samples)); texts += t
    return torch.cat(Rs), torch.cat(Ls), texts

t0 = time.time()
R_before, L_before, texts_before = evaluate(policy)
se_q = lambda x: (x.std() / math.sqrt(len(x))).item()                # SE over questions (std with n − 1)
print(f"held-out reward before training: {R_before.mean():.3f} ± {se_q(R_before.mean(1)):.3f} "
      f"| mean length {L_before.mean():.1f} tokens | {time.time() - t0:.0f} s")

The training loop. Each iteration samples a batch, scores it with the rule, and computes the loss in
two chunks of 16 answers (two whole groups each) to halve the activation memory; the gradients of
the chunks add up to the gradient of the full-batch loss, because each chunk's loss is scaled by its
share of the batch. Logged per iteration: mean reward, mean $k_3$ per token, the fraction of
zero-variance groups, and the mean response length.

In [ ]:
def chunk(ro, a, b):
    return {k: (v[a:b] if torch.is_tensor(v) and v.dim() > 0 else v) for k, v in ro.items()}

opt = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=0.0)
torch.manual_seed(0)
log, t0 = [], time.time()
for it in range(ITERS):
    questions = train_q[it * PROMPTS:(it + 1) * PROMPTS]
    with autocast():
        ro = Q.rollout(policy, tok, [ask(q) for q in questions for _ in range(G)], max_new=MAX_NEW)
    R, _ = rewards_of(ro)
    R, N = R.to(DEVICE), len(R)
    opt.zero_grad(set_to_none=True)
    k3_sum = 0.0
    for c in range(0, N, CHUNK):
        sub = chunk(ro, c, c + CHUNK)
        with torch.no_grad(), autocast():
            ref_logp = Q.token_logps(ref, sub)
        with autocast():
            logp = Q.token_logps(policy, sub)
        A, loss = grpo_terms(R[c:c + CHUNK], logp, ref_logp, sub["mask"], G)
        (loss * len(A) / N).backward()
        k3_sum += float((k3(logp.detach(), ref_logp) * sub["mask"]).sum())
    grad_norm = float(torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0))
    opt.step()
    if it == 0:
        static_gib = Q.gib(torch.cuda.memory_allocated())      # weights + grads + AdamW state + reference
    log.append({"reward": R.mean().item(), "k3": k3_sum / ro["mask"].sum().item(),
                "zero_var": (R.view(-1, G).std(-1) == 0).float().mean().item(),
                "length": ro["mask"].sum(1).float().mean().item(), "grad_norm": grad_norm})
    if it % 10 == 9:
        last = log[-10:]
        print(f"iter {it + 1:2d} | reward {np.mean([l['reward'] for l in last]):.3f} | k3 {log[-1]['k3']:.4f} | "
              f"zero-var groups {np.mean([l['zero_var'] for l in last]):.2f} | "
              f"length {np.mean([l['length'] for l in last]):.1f} | {time.time() - t0:.0f} s")
peak_gib = Q.gib(torch.cuda.max_memory_allocated())
assert log[0]["k3"] < 1e-4                     # π_θ = π_ref before the first step
print(f"k3 at iteration 1 (before any update): {log[0]['k3']:.1e}")
print(f"peak GPU memory {peak_gib:.1f} GiB | after the first step {static_gib:.1f} GiB allocated")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
its = np.arange(1, ITERS + 1)
get = lambda k: np.array([l[k] for l in log])
smooth = lambda x: np.convolve(x, np.ones(5) / 5, mode="valid")
axes[0].plot(its, get("reward"), color=BLUE, alpha=0.35, marker=".")
axes[0].plot(its[4:], smooth(get("reward")), color=BLUE, label="5-iteration mean")
axes[0].set(title="mean reward per batch", xlabel="iteration", ylim=(0, 1.02)); axes[0].legend(fontsize=8, frameon=False)
axes[1].plot(its, get("k3"), color=ORANGE, marker=".")
axes[1].set(title="mean k3 per token (KL to π_ref)", xlabel="iteration")
axes[2].plot(its, get("zero_var"), color=GREEN, marker=".", label="measured (4 groups)")
axes[2].plot(its, zero_signal_prob(get("reward"), G), color=GRAY, ls="--", label="iid formula at batch mean p")
axes[2].set(title="zero-variance groups", xlabel="iteration", ylim=(-0.03, 1.03)); axes[2].legend(fontsize=8, frameon=False)
axes[3].plot(its, get("length"), color=RED, marker=".")
axes[3].set(title="mean response length (tokens)", xlabel="iteration")
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![GRPO on Qwen3-0.6B: reward, k3 KL, zero-variance groups and response length per iteration](../figures/chapter-13/study-path-s4_grpo-05.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

In [ ]:
t0 = time.time()
R_after, L_after, texts_after = evaluate(policy)
diff = R_after.mean(1) - R_before.mean(1)                        # paired by question
print(f"held-out reward: before {R_before.mean():.3f} ± {se_q(R_before.mean(1)):.3f}  →  "
      f"after {R_after.mean():.3f} ± {se_q(R_after.mean(1)):.3f}   "
      f"(paired difference {diff.mean():+.3f} ± {se_q(diff):.3f}, {len(diff)} questions × 2 answers)")
print(f"mean length: {L_before.mean():.1f} → {L_after.mean():.1f} tokens   ({time.time() - t0:.0f} s)")
for j in [0, 5, 11]:
    print(f"\nQ: {test_q[j]}")
    print(f"   before: {textwrap.shorten(texts_before[2 * j], 140)}")
    print(f"   after:  {textwrap.shorten(texts_after[2 * j], 140)}")

**Memory without a value model.** The numbers printed after training are measured on this run. The
static part after the first step holds the policy's float32 weights, gradients and two AdamW moments
(16 bytes per parameter) plus the float32 reference (4 bytes per parameter). PPO on the same batch
would add a **trained value model**: in the usual set-up another copy of the same transformer with a
scalar head, i.e. another 16 bytes per parameter of weights, gradients and optimizer state, plus its
own forward and backward activations on every sampled sequence. The next cell puts the two side by
side; the PPO column is an estimate from the parameter count, not a measurement.

In [ ]:
per_param = {"policy weights (fp32)": 4, "policy gradients": 4, "AdamW moments (2 × fp32)": 8, "reference (fp32)": 4}
expected_static = sum(per_param.values()) * n_params / 2**30
value_static = 16 * n_params / 2**30
print(f"parameters: {n_params / 1e6:.0f} M")
for k, v in per_param.items():
    print(f"   {k:26s} {v * n_params / 2**30:5.1f} GiB")
print(f"GRPO static, expected {expected_static:.1f} GiB | measured after step 1: {static_gib:.1f} GiB "
      f"| measured peak (with activations): {peak_gib:.1f} GiB")
print(f"PPO would add a trained value model: ≈ {value_static:.1f} GiB static (same body, 16 B/param) "
      f"+ its activations → static ≈ {expected_static + value_static:.1f} GiB, "
      f"{(expected_static + value_static) / expected_static:.1f}× GRPO's")
del opt, policy, ref
Q.free()

### What the real run shows

Numbers below are from the reference run on the Spark; your run prints its own.

- **Reward rises, and the zero-variance fraction follows it.** The batch reward, averaged over 10
  iterations, goes 0.35 → 0.48 → 0.88 → 0.92. The fraction of zero-variance groups is 0.38 in the
  first 10 iterations, falls to 0.12 while the reward is near 0.5, and climbs to 0.65 in the last 10,
  when most groups are answered briefly 8 times out of 8: Part A's Predict happening in a real run.
  In the first 10 iterations the measured fraction is well above the iid formula evaluated at the
  batch mean (dashed line), because the four questions in a batch differ in difficulty: a question
  that is always, or never, answered briefly gives a constant group whatever the batch average is.
  Near the end, with most answers short, the two are close. With only 4 groups per batch each point
  moves in steps of 0.25.
- **$k_3$ starts at exactly 0** (π_θ = π_ref before the first step) and grows to about 0.6 nats per
  token by iteration 40: the policy has moved measurably away from the reference. Once most groups
  are constant, the KL term is the only part of the loss with a gradient for them.
- **Length falls**: held-out answers go from 33.4 to 17.2 tokens on average. The rule rewards
  brevity, and that is what moved.
- **Held-out reward** on 96 unseen questions (2 answers each) goes from $0.271\pm0.038$ to
  $0.891\pm0.025$; the paired difference is $+0.620\pm0.042$, about 15 standard errors from zero.
  That establishes that GRPO with only the policy and a frozen reference changed this behaviour on
  new prompts. The memory cell measures 11.2 GiB static (11.1 expected) and a 17.2 GiB peak; a PPO
  value model would add about 8.9 GiB of static memory alone.
- **What it does not establish.** The sample answers show how the rule was satisfied: by dropping
  detail. Some short answers are less accurate than the long ones they replace (in the reference run,
  the BitTorrent question gets a flat "not illegal" with the caveats removed). Nothing here measures
  correctness or helpfulness; a length rule can be satisfied without either, which is the
  reward-hacking problem of S7. One seed and 40 updates say nothing about stability over longer runs.

---

# Summary

| Part | You wrote | Measured / verified |
|---|---|---|
| A | `group_advantages`, `zero_signal_prob` | z-score sums to 0 per group, 0 for constant groups, invariant to $aR+b$; at $p=0.9$, $G=8$, 43% of groups give no signal |
| B | `seq_mean`, `token_mean`, `constant_norm`, `centered_advantages` | per-token weights $1/(N\lvert o_i\rvert)$, $1/\sum\lvert o_j\rvert$, $1/(NL_{\max})$; per-answer mean pushes each token of a long wrong answer 10× less; the std division up-weights nearly-solved and nearly-failed prompts (×1.5 at $k=1$ vs $k=4$) |
| C | `k1`, `k3`, `kl_exact`, `k3_surrogate` | both unbiased; $k_3$ is 5× less noisy near $\pi_{\text{ref}}$ and 10× noisier far from it; $k_3$'s expected gradient is $\nabla\mathrm{KL}(\pi_{\text{ref}}\|\pi)=\pi-\pi_{\text{ref}}$; $k_1$ in the loss has zero expected gradient |
| D | `grpo_loss` | at a fresh batch $\rho=1$, loss 0, gradient nonzero; reusing a batch for 8 steps clips ≈ 60% of tokens but ratios still leave $[0.8, 1.2]$ |
| E | `reinforce_loss` | GRPO is invariant to a reward offset (same run to $10^{-14}$); REINFORCE with $R+1$ learns far slower; with $R\in\{0,1\}$ no-baseline REINFORCE was slightly ahead of GRPO in this toy |
| F | `length_reward`, `grpo_terms` | Qwen3-0.6B, policy + frozen reference only: held-out reward 0.271 → 0.891, paired difference $+0.620\pm0.042$; length 33 → 17 tokens; $k_3$ 0 → 0.6 nats/token; zero-variance groups 0.38 → 0.12 → 0.65; peak 17.2 GiB, a value model would add ≈ 8.9 GiB static |

**Evidence.** The identities in Parts A–D are exact and asserted on fixed tensors. The toy comparison
in Part E is averaged over 8 seeds with standard errors and holds for this tabular policy with Adam
only. Part F establishes, for one run, that the GRPO update implemented here changes Qwen3-0.6B's
behaviour on held-out prompts in the direction of the reward, with the diagnostics behaving as Parts
A–C predict. It does not establish any gain in answer quality, correctness or helpfulness (the reward
doesn't measure them, and the samples show generic short answers), nor anything about longer runs,
other seeds, other rewards or larger models. Mechanics verified; no capability claim.

**Next stage.** [S5 · RLVR](s5_rlvr.ipynb) keeps this GRPO update and replaces the length rule with a
real **verifier**: a program that checks whether an answer is correct. Discussions to revisit:
[D33–D35](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md).